In [2]:
import json

tweets = []

with open("Combined_tweets.json", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
            tweets.append(obj)
        except json.JSONDecodeError:
            pass  # skip broken lines


In [3]:
cleaned = []

for item in tweets:
    if isinstance(item, list):
        cleaned.extend(item)
    elif isinstance(item, dict):
        cleaned.append(item)


In [4]:
final_tweets = [
    {
        "id": t.get("id"),
        "text": t.get("text", "")
    }
    for t in cleaned
    if "text" in t
]


In [5]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)       # URLs
    text = re.sub(r"@\w+", "", text)           # mentions
    text = re.sub(r"#\w+", "", text)           # hashtags
    text = re.sub(r"[^\w\s]", "", text)        # punctuation
    return text.strip()


In [6]:
texts = [clean_text(t["text"]) for t in final_tweets]

In [8]:
from collections import Counter

words = []
for text in texts:
    words.extend(text.split())

c = Counter(words)

for word, count in c.most_common(20):
    print(f"{word}: {count}")



elon: 208
to: 191
musk: 187
you: 136
mr: 133
the: 130
for: 126
this: 113
your: 99
in: 82
is: 82
of: 80
follow: 78
signed: 78
a: 68
and: 63
or: 63
on: 63
back: 56
package: 55


In [10]:
# N-Gram
texts = [clean_text(t["text"]) for t in final_tweets]

# Generate bigrams
from collections import Counter

def ngrams(tokens, n):
    return zip(*[tokens[i:] for i in range(n)])

bigram_counter = Counter()

for text in texts:
    tokens = text.split()
    bigram_counter.update(ngrams(tokens, 2))

for gram, count in bigram_counter.most_common(10):
    print(" ".join(gram), ":", count)


elon musk : 184
mr elon : 133
your package : 55
this is : 54
package signed : 53
follow back : 50
if you : 46
to get : 46
for all : 44
get your : 44


In [12]:
trigram_counter = Counter()

for text in texts:
    tokens = text.split()
    trigram_counter.update(ngrams(tokens, 3))
filtered = {
    gram: count
    for gram, count in bigram_counter.items()
    if count >= 3   # appears at least 3 times
}
for gram, count in trigram_counter.most_common(10):
    print(" ".join(gram), ":", count)


mr elon musk : 126
your package signed : 53
to get your : 44
get your package : 44
approved by mr : 43
by mr elon : 43
this is the : 43
is the only : 43
the only way : 43
only way to : 43


In [13]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    ngram_range=(2, 3),
    stop_words="english",
    min_df=3
)

X = vectorizer.fit_transform(texts)

ngrams = vectorizer.get_feature_names_out()
counts = X.sum(axis=0).A1

top = sorted(zip(ngrams, counts), key=lambda x: x[1], reverse=True)[:10]

for gram, count in top:
    print(f"{gram}: {count}")


elon musk: 184
mr elon: 133
mr elon musk: 126
package signed: 53
appreciation fans: 43
appreciation fans mr: 43
approved mr: 43
approved mr elon: 43
comments package: 43
fans mr: 43


In [14]:
# NB Theorem

def label_tweet(text):
    return 1 if "python" in text else 0

#texts = [clean_text(t["text"]) for t in final_tweets]
labels = [label_tweet(text) for text in texts]
vectorizer = CountVectorizer(
    ngram_range=(1, 2),   # unigrams + bigrams
    stop_words="english",
    min_df=2              # ignore very rare terms
)

X = vectorizer.fit_transform(texts)
y = labels

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train Naive Bayes classifier
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))



Accuracy: 0.8823529411764706
              precision    recall  f1-score   support

           0       1.00      0.88      0.94        33
           1       0.20      1.00      0.33         1

    accuracy                           0.88        34
   macro avg       0.60      0.94      0.63        34
weighted avg       0.98      0.88      0.92        34



In [15]:
# Test with new tweets
new_tweets = [
    "I love Python for data science",
    "Donald Trump rallies supporters again"
]

new_tweets_clean = [clean_text(t) for t in new_tweets]
X_new = vectorizer.transform(new_tweets_clean)

predictions = model.predict(X_new)

for tweet, pred in zip(new_tweets, predictions):
    print(tweet, "→", "Python-related" if pred == 1 else "Other")


I love Python for data science → Python-related
Donald Trump rallies supporters again → Other


In [16]:
# Evaluation using the F1-score

from sklearn.metrics import f1_score

f1 = f1_score(y_test, y_pred)
print("F1-score:", f1)

F1-score: 0.3333333333333333
